# Hybrid-Locality: Do Linear Layers in Hybrid LMs Use Long Context?
**Project:** hybrid-locality (P1 Intervention Study from Research Scout)
**Branch:** `feature/hybrid-locality`
**Target Venue:** ICML 2027 / NeurIPS 2027
**Lead Researcher:** Jaime Jarauta
**One-Click Execution:** Press **Runtime -> Run all** to clone, setup, execute, sync to W&B, and auto-unassign GPU.

In [ ]:
# Step 1: Self-Healing Environment Setup, Bulletproof Credit Guard & Colab Secrets Activation
import os, sys, signal, subprocess
from IPython import get_ipython

def auto_teardown(signum=None, frame=None):
    print("\n[Credit Guard] Unhandled error, interrupt, or completion! Syncing W&B and unassigning GPU...")
    try:
        subprocess.run("wandb sync --include-offline 2>/dev/null || true", shell=True)
    except Exception:
        pass
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass

def custom_exc_handler(self, etype, value, tb, tb_offset=None):
    self.showtraceback((etype, value, tb), tb_offset=tb_offset)
    auto_teardown()
    return []

ip = get_ipython()
if ip is not None:
    ip.set_custom_exc((Exception,), custom_exc_handler)

# Activate API Secrets automatically from Colab Userdata / Secrets panel
try:
    from google.colab import userdata
    for k in ["HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "hf_token"]:
        try:
            tok = userdata.get(k)
            if tok:
                os.environ["HF_TOKEN"] = tok
                os.environ["HUGGING_FACE_HUB_TOKEN"] = tok
                print(f"[Secrets] Successfully loaded Hugging Face token from Colab Secrets ({k}).")
                break
        except Exception:
            pass
    for k in ["WANDB_API_KEY", "wandb_api_key", "WANDB_KEY"]:
        try:
            key = userdata.get(k)
            if key:
                os.environ["WANDB_API_KEY"] = key
                print(f"[Secrets] Successfully loaded Weights & Biases API key from Colab Secrets ({k}).")
                break
        except Exception:
            pass
except ImportError:
    pass

print("[Ready] Credit guard active. L4 GPU default configured. VM will self-terminate on error or completion.")


In [ ]:
# Step 2: Clone Repository & Install Dependencies
import os, sys, subprocess
from pathlib import Path
import torch

assert torch.cuda.is_available(), "Switch Colab runtime to GPU (L4 24GB recommended): Runtime -> Change runtime type -> GPU (L4)"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB | BF16: {torch.cuda.is_bf16_supported()}")

repo_url = "https://github.com/jaimejarauta/decoupled-delta-net.git"
branch = "feature/hybrid-locality"
target_dir = "/content/hybrid-locality"

%cd /content
if not os.path.exists(target_dir):
    print(f"Cloning hybrid-locality from {repo_url} (branch: {branch})...")
    subprocess.run(["git", "clone", "-b", branch, repo_url, target_dir], check=True)
else:
    print(f"Updating hybrid-locality to latest {branch}...")
    subprocess.run(["git", "-C", target_dir, "fetch", "origin", branch], check=True)
    subprocess.run(["git", "-C", target_dir, "checkout", branch], check=True)
    subprocess.run(["git", "-C", target_dir, "reset", "--hard", f"origin/{branch}"], check=True)
    subprocess.run(["git", "-C", target_dir, "pull", "origin", branch], check=True)

%cd /content/hybrid-locality
!pip install -q -e .
!pip install -q "transformers>=5.0" "datasets>=3.0" accelerate pytest wandb

# Auto-login to Hugging Face and W&B if secrets were loaded
if os.environ.get("HF_TOKEN"):
    try:
        import huggingface_hub
        huggingface_hub.login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
        print("[Auth] Hugging Face Hub successfully logged in.")
    except Exception as e:
        print(f"[Auth] HF login note: {e}")
if os.environ.get("WANDB_API_KEY"):
    try:
        import wandb
        wandb.login(key=os.environ["WANDB_API_KEY"])
        print("[Auth] Weights & Biases successfully logged in.")
    except Exception as e:
        print(f"[Auth] W&B login note: {e}")

if "/content/hybrid-locality/src" not in sys.path:
    sys.path.insert(0, "/content/hybrid-locality/src")
os.environ["PYTHONPATH"] = "/content/hybrid-locality/src:" + os.environ.get("PYTHONPATH", "")
print("[Ready] hybrid-locality environment ready on GPU.")


In [ ]:
# Step 3: Run Fast CPU Pre-Flight Checks (Exactness, Locality, and Gate Logic)
%cd /content/hybrid-locality
!pytest -q tests/test_window.py


In [ ]:
# Step 4: Execute Long-Context Needle & LongPPL Intervention Study
%cd /content/hybrid-locality
import os
os.makedirs("results", exist_ok=True)

# 1. Evaluate Qwen3.5-0.8B (18 Gated DeltaNet + 6 Attention layers)
print("=" * 70)
print("EVALUATING MODEL 1: Qwen/Qwen3.5-0.8B (Gated DeltaNet Hybrid)")
print("=" * 70)
!python scripts/run_kill.py run --model Qwen/Qwen3.5-0.8B --lengths 8192 16384 --n 50 --docs 10

# 2. Evaluate Falcon-H1-0.5B-Base (Mamba-2 + Attention hybrid)
print("=" * 70)
print("EVALUATING MODEL 2: tiiuae/Falcon-H1-0.5B-Base (Mamba-2 Hybrid)")
print("=" * 70)
!python scripts/run_kill.py run --model tiiuae/Falcon-H1-0.5B-Base --lengths 8192 --n 50 --docs 10

# 3. Generate Pre-Registered Verdict & Bootstrap Confidence Intervals
print("=" * 70)
print("PRE-REGISTERED HYBRID LOCALITY VERDICT & REPORT")
print("=" * 70)
!python scripts/run_kill.py report


In [ ]:
# Step 5: Sync Telemetry & Unassign GPU (Zero Credit Leakage)
print("[Credit Guard] Sweep complete! Syncing W&B and unassigning GPU...")
try:
    auto_teardown()
except NameError:
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print(f"Teardown note: {e}")
